## The container lifecycle & restart policies

Module 01 sketched the lifecycle; here's the fuller picture plus how production keeps a container alive. The states and the verbs that move between them:

```
  create        start         stop / kill / exits          rm
 image ─► created ─► running ─────────────────► stopped ─────► gone
                       │  ▲                         │
               pause   │  │ unpause     start ──────┘
                       ▼  │
                     paused
```

- **`create`** makes a container from an image without starting it (allocates the writable layer, config, name) — rare on its own, but it's the half of `run` you don't usually see.
- **`start` / `stop` / `restart` / `rm`** are the module-01 verbs. **`pause` / `unpause`** freeze and thaw every process via the freezer cgroup — useful to momentarily halt a container without losing its state.

**Restart policies** — by default an exited container stays exited, but a production service must survive crashes *and* host reboots. `--restart` sets that:

| Policy | Behaviour |
|---|---|
| **`no`** *(default)* | never restart. |
| **`on-failure[:N]`** | restart only on a non-zero exit; optional max retries. |
| **`always`** | restart on any exit *and* when the daemon starts — comes back after a reboot (even if you `docker stop` it, it returns on next boot). |
| **`unless-stopped`** | like `always`, but a manual `docker stop` sticks across daemon restarts. **The usual production choice.** |

```bash
docker run -d --restart=unless-stopped --name api myorg/api:1.0
```

Details that bite: the policy only applies *after* the first successful start (a container that never starts isn't retried); backoff is **exponential**, capped near a minute, to avoid hammering the host in a crash loop; and the restart counter resets after 10s of healthy running.